<a href="https://colab.research.google.com/github/AnanyaAsthana/Hadoop-CUDA-Lab/blob/main/HadoopCudaLab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Wed Jan 14 10:08:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
%%writefile device_info.cu
#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    int deviceCount = 0;
    cudaGetDeviceCount(&deviceCount);

    if (deviceCount == 0) {
        printf("No CUDA-enabled devices found.\n");
        return 0;
    }

    int driverVersion = 0;
    int runtimeVersion = 0;

    cudaDriverGetVersion(&driverVersion);
    cudaRuntimeGetVersion(&runtimeVersion);

    printf("CUDA Driver Version  : %d.%d\n",
           driverVersion / 1000, (driverVersion % 100) / 10);
    printf("CUDA Runtime Version : %d.%d\n\n",
           runtimeVersion / 1000, (runtimeVersion % 100) / 10);

    for (int dev = 0; dev < deviceCount; dev++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, dev);

        printf("Device %d: %s\n", dev, prop.name);
        printf("  Compute Capability : %d.%d\n",
               prop.major, prop.minor);
        printf("  Total Global Memory: %zu bytes\n",
               prop.totalGlobalMem);
        printf("\n");
    }

    return 0;
}


Writing device_info.cu


In [ ]:
!nvcc device_info.cu -o device_info


In [ ]:
!./device_info


CUDA Driver Version  : 12.4
CUDA Runtime Version : 12.5

Device 0: Tesla T4
  Compute Capability : 7.5
  Total Global Memory: 15828320256 bytes



In [ ]:
%%writefile hello.cu
#include <stdio.h>
#include <cuda_runtime.h>

// GPU kernel
__global__ void helloFromGPU() {
    printf("Hello World from GPU!\n");
}

int main() {
    // CPU output
    printf("Hello World from CPU!\n");

    // Launch GPU kernel
    helloFromGPU<<<1, 1>>>();

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    return 0;
}


Writing hello.cu


In [ ]:
!nvcc hello.cu -o hello


In [ ]:
!./hello


Hello World from CPU!


In [ ]:
%%writefile hello_fixed.cu
#include <stdio.h>
#include <cuda_runtime.h>

// GPU kernel
__global__ void helloFromGPU(int *flag) {
    flag[0] = 1;   // GPU writes to memory
}

int main() {
    printf("Hello World from CPU!\n");

    int h_flag = 0;
    int *d_flag;

    cudaMalloc((void**)&d_flag, sizeof(int));
    cudaMemcpy(d_flag, &h_flag, sizeof(int), cudaMemcpyHostToDevice);

    helloFromGPU<<<1,1>>>(d_flag);
    cudaDeviceSynchronize();

    cudaMemcpy(&h_flag, d_flag, sizeof(int), cudaMemcpyDeviceToHost);

    if (h_flag == 1) {
        printf("Hello World from GPU!\n");
    }

    cudaFree(d_flag);
    return 0;
}


Writing hello_fixed.cu


In [ ]:
!nvcc hello_fixed.cu -o hello_fixed
!./hello_fixed


Hello World from CPU!
